# Bộ nhớ khi train: đo, cộng dồn gradient, checkpointing

Gặp `CUDA out of memory`, người mới giảm batch size rồi thôi. Người có kinh nghiệm hỏi trước: **bộ nhớ đang nằm ở
đâu**: tham số, gradient, trạng thái optimizer, hay activation giữ cho backward? Mỗi chỗ có cách chữa riêng. Bạn
viết công cụ đo activation thật bằng `saved_tensors_hooks`, đếm trạng thái train, cài gradient accumulation đúng
cả khi batch không chia hết, và activation checkpointing theo đoạn; rồi đo trên GPU bằng
`torch.cuda.max_memory_allocated`, `torch.profiler` và memory snapshot.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm và lớp. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật, đo, in; test bỏ qua chúng.
- Phần đo activation, trạng thái train và accumulation chạy trên CPU. Các cell ghi **(GPU)** cần kernel `ai-gpu`
  (xem `setup/gpu/README.md`); với env gốc chúng chỉ in lời nhắc. Cả notebook dùng dưới 1.5 GB VRAM.
- Chấm: `uv run pytest tracks/ai-engineer/modules/02-pytorch-deep-dive/exercises/intermediate/02-memory-accumulation-checkpointing -q`

In [ ]:
# explore: giới hạn luồng CPU, tìm gốc repo, kiểm CIFAR-10 (137 MB, tải một lần bằng env gốc)
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")   # đặt TRƯỚC khi import torch; GPU dùng chung, CPU cũng vậy
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
MODULE = ROOT / "tracks/ai-engineer/modules/02-pytorch-deep-dive"
CIFAR = ROOT / "data/raw/cifar10"
if not (CIFAR.is_dir() and any(CIFAR.iterdir())):
    # kernel ai-gpu không có thư viện tải dữ liệu: khi đó chạy ở terminal `uv run python data/download_dataset.py cifar10`
    sys.path.insert(0, str(ROOT))
    from data.download_dataset import ensure

    ensure("cifar10")
print(CIFAR, sorted(p.name for p in (CIFAR / "plain_text").iterdir()))

In [ ]:
import tempfile
import time

import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.checkpoint import checkpoint

## 1. Activation giữ cho backward

Mỗi op trong forward lưu lại vài tensor để backward dùng (`Linear` lưu input, `ReLU` lưu output, `BatchNorm` lưu
input và thống kê...). `torch.autograd.graph.saved_tensors_hooks(pack, unpack)` cho bạn thấy từng tensor đó lúc
nó được lưu. Đếm theo **storage** chứ không theo tensor: nhiều op lưu cùng một tensor, hoặc lưu view của nhau.

In [ ]:
def saved_tensor_bytes(model: nn.Module, x: torch.Tensor) -> int:
    """Chạy model(x) một lần và trả TỔNG SỐ BYTE mà autograd giữ lại cho backward.

    Dùng torch.autograd.graph.saved_tensors_hooks: pack hook thấy mọi tensor được lưu cho backward.
    Đếm theo storage (t.untyped_storage()): nhiều tensor là view của cùng storage chỉ tính một lần, tính
    nbytes() của cả storage; bỏ qua storage của tham số model (trọng số tồn tại dù có train hay không).
    Không có gì được lưu (no_grad, eval với inference) thì trả 0.
    """
    # TODO
    raise NotImplementedError

## 2. Trạng thái train

Phần còn lại của bộ nhớ train là thứ sống suốt quá trình: tham số, gradient, trạng thái optimizer. Đếm byte thật
từ các tensor, không nhân công thức, để thấy `AdamW` giữ những gì (kể cả `step`).

In [ ]:
def state_bytes(model: nn.Module, optimizer: torch.optim.Optimizer) -> dict[str, int]:
    """Đếm byte THẬT đang chiếm (không ước lượng theo công thức):

    {"params": byte của mọi tham số, "grads": byte của mọi .grad khác None,
     "optimizer": byte của mọi tensor trong optimizer.state (momentum, exp_avg, exp_avg_sq, step...)}.
    """
    # TODO
    raise NotImplementedError

## 3. Gradient accumulation

Batch 256 không vừa bộ nhớ thì chạy 4 micro-batch 64 rồi mới `step`. Gradient cộng dồn trong `.grad` (đó là lý
do PyTorch không tự xoá gradient). Chỗ hay sai: chia loss cho **số micro-batch** thay vì theo **số mẫu**, sai khi
micro-batch cuối nhỏ hơn.

In [ ]:
def accumulate_step(model: nn.Module, x: torch.Tensor, y: torch.Tensor, optimizer: torch.optim.Optimizer,
                    micro_batch: int) -> float:
    """Một bước optimizer cho cả batch (x, y) nhưng forward/backward theo từng micro-batch tối đa `micro_batch`
    mẫu (micro-batch cuối có thể nhỏ hơn), cộng dồn gradient, rồi step đúng MỘT lần.

    Gradient cuối phải bằng gradient của F.cross_entropy (trung bình) trên cả batch, kể cả khi n không chia
    hết cho micro_batch. Trả loss trung bình của cả batch (float).
    """
    # TODO
    raise NotImplementedError

## 4. Activation checkpointing

Không lưu activation bên trong một đoạn, chỉ lưu input của đoạn; khi backward, chạy lại forward của đoạn đó. Đổi
bộ nhớ lấy tính toán. `use_reentrant=False` là cài đặt PyTorch khuyên dùng; nó cũng giữ trạng thái RNG để Dropout
khi tính lại sinh đúng mask cũ.

In [ ]:
class CheckpointedSequential(nn.Module):
    """Như nn.Sequential(*layers) nhưng activation checkpointing theo đoạn.

    - self.layers = nn.ModuleList(layers) (state_dict giống nn.Sequential có tiền tố "layers.").
    - Chia layers thành các đoạn LIÊN TIẾP, mỗi đoạn ceil(len(layers) / segments) layer (đoạn cuối có thể ngắn hơn).
    - Khi self.training và grad đang bật: mỗi đoạn chạy qua torch.utils.checkpoint.checkpoint(..., use_reentrant=False),
      để chỉ input của từng đoạn được giữ lại, phần bên trong tính lại khi backward. Ngược lại chạy bình thường.
    - Output và gradient phải giống hệt bản không checkpoint, kể cả khi có Dropout (cùng seed).
    """

    def __init__(self, layers: list[nn.Module], segments: int):
        super().__init__()
        # TODO
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO
        raise NotImplementedError

## 5. Đo trên ResNet-9 và CIFAR-10

Cell đầu dùng lại ResNet-9 của bài 01. Các con số activation chỉ phụ thuộc shape nên giống nhau trên mọi máy.

In [ ]:
# explore: ResNet-9 như bài 01 (độ rộng w), chọn thiết bị
def conv_bn(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True))


class Residual(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.f = nn.Sequential(conv_bn(c, c), conv_bn(c, c))

    def forward(self, x):
        return x + self.f(x)


def resnet9(w=32, n_classes=10):
    return nn.Sequential(
        conv_bn(3, w),
        conv_bn(w, 2 * w), nn.MaxPool2d(2), Residual(2 * w),
        conv_bn(2 * w, 4 * w), nn.MaxPool2d(2),
        conv_bn(4 * w, 8 * w), nn.MaxPool2d(2), Residual(8 * w),
        nn.AdaptiveMaxPool2d(1), nn.Flatten(), nn.Linear(8 * w, n_classes),
    )


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MEAN = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1).to(DEVICE)
STD = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1).to(DEVICE)
print(DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "(đo GPU cần kernel ai-gpu)")

In [ ]:
# explore: một batch CIFAR-10 thật (data.py của bạn ở Basic 03, chưa xong thì lời giải mẫu); ResNet-9 như bài 01
import importlib.util

for path in [MODULE / "exercises/basic/03-fast-dataloader/starter/data.py", MODULE / "solutions/basic/03-fast-dataloader/data.py"]:
    spec = importlib.util.spec_from_file_location(f"basic03_{path.parent.name}", path)
    data_mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(data_mod)
    try:
        x_test, y_test = data_mod.decode_cifar_parquet(CIFAR / "plain_text/test-00000-of-00001.parquet")
        break
    except NotImplementedError:
        continue
print("data.py:", path.relative_to(ROOT), "| test", tuple(x_test.shape))

In [ ]:
# explore: bộ nhớ activation của ResNet-9 (W=32), batch 128, đo bằng saved_tensor_bytes của bạn. Chỉ phụ thuộc
# shape nên giống nhau trên CPU và GPU.
MiB = 2**20
torch.manual_seed(0)
net = resnet9(32)
xb = ((x_test[:128].float() / 255 - MEAN.cpu()) / STD.cpu())
print(f"fp32                 : {saved_tensor_bytes(net, xb) / MiB:7.1f} MiB")
with torch.autocast("cpu", dtype=torch.bfloat16):
    print(f"autocast bf16        : {saved_tensor_bytes(net, xb) / MiB:7.1f} MiB")
with torch.no_grad():
    print(f"no_grad              : {saved_tensor_bytes(net, xb) / MiB:7.1f} MiB")
for segments in (12, 4, 2, 1):
    ck = CheckpointedSequential(list(resnet9(32)), segments)
    print(f"checkpoint {segments:2d} đoạn    : {saved_tensor_bytes(ck, xb) / MiB:7.1f} MiB")
print("layer:", [type(m).__name__ for m in net])

In [ ]:
# explore: bộ nhớ "trạng thái train" thật: tham số, gradient, trạng thái optimizer, sau một bước
for opt_name in ("SGD", "SGD momentum", "AdamW"):
    torch.manual_seed(0)
    net = resnet9(32)
    opt = {"SGD": lambda p: torch.optim.SGD(p, lr=0.1),
           "SGD momentum": lambda p: torch.optim.SGD(p, lr=0.1, momentum=0.9),
           "AdamW": lambda p: torch.optim.AdamW(p, lr=1e-3)}[opt_name](net.parameters())
    F.cross_entropy(net(xb[:8]), y_test[:8]).backward()
    opt.step()
    sb = state_bytes(net, opt)
    n = sum(p.numel() for p in net.parameters())
    print(f"{opt_name:13s} params {sb['params'] / MiB:5.2f} MiB | grads {sb['grads'] / MiB:5.2f} | optimizer "
          f"{sb['optimizer'] / MiB:5.2f} | tổng {sum(sb.values()) / n:.1f} byte/tham số")

In [ ]:
# explore: gradient accumulation 4 x 64 so với một batch 256. Có BatchNorm thì có còn tương đương không?
def grads_after(model, micro):
    snapshot = {k: v.clone() for k, v in model.state_dict().items()}
    opt = torch.optim.SGD(model.parameters(), lr=0.0)            # lr 0: step không đổi trọng số, chỉ để giữ gradient
    accumulate_step(model, xb256, y256, opt, micro)
    g = torch.cat([p.grad.flatten() for p in model.parameters()])
    model.load_state_dict(snapshot)                               # trả lại running stats của BN
    return g


def gn_resnet9(w=32):
    net = resnet9(w)
    for name, m in list(net.named_modules()):
        for child_name, child in list(m.named_children()):
            if isinstance(child, nn.BatchNorm2d):
                setattr(m, child_name, nn.GroupNorm(8, child.num_features))
    return net


xb256, y256 = ((x_test[:256].float() / 255 - MEAN.cpu()) / STD.cpu()), y_test[:256]
for name, make in [("BatchNorm", resnet9), ("GroupNorm(8)", gn_resnet9)]:
    torch.manual_seed(0)
    net = make(32)
    full, acc = grads_after(net, 256), grads_after(net, 64)
    print(f"{name:12s} |g_256 - g_4x64| / |g_256| = {(full - acc).norm() / full.norm():.2e}")

## 6. Trên GPU: bộ nhớ theo từng giai đoạn, batch size và checkpoint

`torch.cuda.memory_allocated()` là byte tensor đang sống; `max_memory_allocated()` là đỉnh kể từ lần reset.
`nvidia-smi` báo nhiều hơn vì caching allocator giữ lại block đã giải phóng, cộng context CUDA (vài trăm MB).

In [ ]:
# explore (GPU): bộ nhớ CUDA theo từng giai đoạn của một bước train ResNet-9 W=64, batch 256, AdamW
def cuda_mib():
    return torch.cuda.memory_allocated() / MiB


if DEVICE == "cuda":
    torch.cuda.empty_cache()
    base = cuda_mib()
    torch.manual_seed(0)
    net = resnet9(64).to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), lr=1e-3)
    x256 = ((x_test[:256].to(DEVICE).float() / 255 - MEAN) / STD)
    y = y_test[:256].to(DEVICE)
    torch.cuda.reset_peak_memory_stats()
    steps = {"model": cuda_mib() - base}
    loss = F.cross_entropy(net(x256), y)
    steps["sau forward"] = cuda_mib() - base
    loss.backward()
    steps["sau backward"] = cuda_mib() - base
    opt.step()
    steps["sau step"] = cuda_mib() - base
    opt.zero_grad(set_to_none=True)
    del loss
    steps["zero_grad, bỏ loss"] = cuda_mib() - base
    print(" | ".join(f"{k}: {v:.0f} MiB" for k, v in steps.items()), f"| peak {torch.cuda.max_memory_allocated() / MiB - base:.0f} MiB")
    del net, opt
else:
    print("Cần GPU (kernel ai-gpu) để đo bộ nhớ CUDA; trên CPU xem saved_tensor_bytes ở trên.")

In [ ]:
# explore (GPU): peak memory và thời gian một bước theo batch size, có/không checkpoint, fp32/bf16 (W=64)
def bench(batch, segments, amp_dtype, steps=10):
    torch.manual_seed(0)
    layers = list(resnet9(64))
    net = (CheckpointedSequential(layers, segments) if segments else nn.Sequential(*layers)).to(DEVICE)
    net = net.to(memory_format=torch.channels_last)
    opt = torch.optim.SGD(net.parameters(), lr=0.01, momentum=0.9)
    xb_ = torch.randn(batch, 3, 32, 32, device=DEVICE).contiguous(memory_format=torch.channels_last)
    yb_ = torch.randint(0, 10, (batch,), device=DEVICE)

    def step():
        with torch.autocast(DEVICE, dtype=amp_dtype or torch.float32, enabled=amp_dtype is not None):
            loss = F.cross_entropy(net(xb_), yb_)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()

    step(); step()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    for _ in range(steps):
        step()
    torch.cuda.synchronize()
    ms = (time.perf_counter() - start) / steps * 1000
    peak = torch.cuda.max_memory_allocated() / MiB
    del net, opt
    torch.cuda.empty_cache()
    return ms, peak


if DEVICE == "cuda":
    for amp_dtype in (None, torch.bfloat16):
        for batch in (64, 128, 256):
            row = [f"{'bf16' if amp_dtype else 'fp32'} batch {batch:3d}"]
            for segments in (None, 4, 1):
                ms, peak = bench(batch, segments, amp_dtype)
                row.append(f"{'không ckpt' if segments is None else f'ckpt {segments}'}: {peak:5.0f} MiB {ms:5.1f} ms")
            print(" | ".join(row))
else:
    print("Cần GPU (kernel ai-gpu).")

## 7. Profiler và memory snapshot

`torch.profiler` cho biết op nào tốn thời gian và op nào cấp phát nhiều; trace mở được bằng Perfetto. Memory snapshot
ghi lại từng lần cấp phát kèm stack trace, xem trên `pytorch.org/memory_viz`: công cụ chính khi điều tra OOM thật.

In [ ]:
# explore: torch.profiler cho 3 bước train (ResNet-9 W=32, batch 128): op nào tốn thời gian, op nào cấp phát nhiều nhất
from torch.profiler import ProfilerActivity, profile

torch.manual_seed(0)
net = resnet9(32).to(DEVICE)
opt = torch.optim.SGD(net.parameters(), lr=0.01, momentum=0.9)
xp = ((x_test[:128].to(DEVICE).float() / 255 - MEAN) / STD)
yp = y_test[:128].to(DEVICE)
activities = [ProfilerActivity.CPU] + ([ProfilerActivity.CUDA] if DEVICE == "cuda" else [])
with profile(activities=activities, profile_memory=True, record_shapes=True) as prof:
    for _ in range(3):
        loss = F.cross_entropy(net(xp), yp)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
sort_time = "self_device_time_total" if DEVICE == "cuda" else "self_cpu_time_total"
sort_mem = "self_device_memory_usage" if DEVICE == "cuda" else "self_cpu_memory_usage"
print(prof.key_averages().table(sort_by=sort_time, row_limit=8))
print(prof.key_averages().table(sort_by=sort_mem, row_limit=6))
trace = Path(tempfile.gettempdir()) / "m02_trace.json"
prof.export_chrome_trace(str(trace))
print("Mở", trace, "bằng https://ui.perfetto.dev hoặc chrome://tracing")

In [ ]:
# explore (GPU): memory snapshot: lịch sử từng lần cấp phát, xem bằng https://pytorch.org/memory_viz
if DEVICE == "cuda":
    torch.cuda.memory._record_memory_history(max_entries=100_000)
    net = resnet9(64).to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), lr=1e-3)
    x256 = ((x_test[:256].to(DEVICE).float() / 255 - MEAN) / STD)
    for _ in range(3):
        loss = F.cross_entropy(net(x256), y_test[:256].to(DEVICE))
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
    snap = Path(tempfile.gettempdir()) / "m02_snapshot.pickle"
    torch.cuda.memory._dump_snapshot(str(snap))
    torch.cuda.memory._record_memory_history(enabled=None)
    print("Kéo", snap, "vào https://pytorch.org/memory_viz (tab Active Memory Timeline)")
    del net, opt, loss
else:
    print("Memory snapshot chỉ có cho CUDA.")

## 8. Đọc kết quả

Trước khi làm trắc nghiệm trong app, tự trả lời:

1. Với ResNet-9 batch 128, activation chiếm bao nhiêu so với trạng thái train (tham số + gradient + AdamW)? Vậy
   giảm batch size cứu được phần nào, và không cứu được phần nào?
2. Vì sao autocast bf16 không giảm activation đúng một nửa?
3. Checkpoint 1 đoạn giữ ít nhất nhưng peak memory trên GPU không giảm tương ứng. Phần nào vẫn phải nằm trong bộ nhớ
   lúc backward? Đổi lại tốn bao nhiêu thời gian?
4. Vì sao gradient accumulation khớp gần như tuyệt đối với GroupNorm mà lệch rõ với BatchNorm?